# 中国股票市场概览数据分析

**分析范围**：1990–2026 年沪深北三大交易所上市公司数量统计、板块结构与行业分布

**数据来源**：上交所、深交所、北交所官网统计数据及中国上市公司协会公开数据

**内容结构**：
1. 环境准备
2. 数据构建（年度历史 / 交易所概况 / 行业分布）
3. 统计分析（关键指标 / 阶段分析 / 行业集中度）
4. 数据可视化（7张图表）
5. 汇总表格展示
6. 导出 Excel 综合报告
7. 分析结论

---

## 1. 环境准备

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 中文字体设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Noto Sans CJK SC', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# 创建目录结构
for d in ['data', 'output/figures', 'output/reports']:
    os.makedirs(d, exist_ok=True)

print('✅ 环境准备完成')
print(f'📅 当前日期: {datetime.now().strftime("%Y-%m-%d")}')


## 2. 数据构建

数据来源于上交所、深交所、北交所官网及中国上市公司协会公开发布的统计数据。

> **注**：
> - 中小板已于 **2021 年 6 月 21 日** 正式并入深交所主板，之后不再单独统计
> - 科创板于 **2019 年 7 月** 正式开市
> - 北交所于 **2021 年 11 月** 正式开市
> - 如需实时数据，可在网络畅通环境下将本模块替换为 `akshare` 或 `tushare` API 调用

### 2.1 年度历史数据（1990–2026）

In [ ]:
# 各年度各交易所各板块上市公司数量
# 列：year, total, sse_main(上交主板), sse_star(科创板), szse_main(深交主板),
#       szse_sme(中小板,2004-2020), szse_gem(创业板,2009+), bse(北交所,2021+)
annual_records = [
    (1990,   10,    8,   0,    2,    0,    0,   0),
    (1991,   14,    8,   0,    6,    0,    0,   0),
    (1992,   53,   29,   0,   24,    0,    0,   0),
    (1993,  183,  106,   0,   77,    0,    0,   0),
    (1994,  291,  171,   0,  120,    0,    0,   0),
    (1995,  323,  188,   0,  135,    0,    0,   0),
    (1996,  530,  293,   0,  237,    0,    0,   0),
    (1997,  745,  383,   0,  362,    0,    0,   0),
    (1998,  851,  438,   0,  413,    0,    0,   0),
    (1999,  949,  484,   0,  463,    0,    0,   0),
    (2000, 1088,  572,   0,  514,    0,    0,   0),
    (2001, 1154,  646,   0,  508,    0,    0,   0),
    (2002, 1224,  715,   0,  509,    0,    0,   0),
    (2003, 1287,  780,   0,  507,    0,    0,   0),
    (2004, 1377,  837,   0,  470,   70,    0,   0),
    (2005, 1381,  834,   0,  435,  112,    0,   0),
    (2006, 1434,  842,   0,  436,  156,    0,   0),
    (2007, 1550,  860,   0,  460,  230,    0,   0),
    (2008, 1625,  864,   0,  465,  296,    0,   0),
    (2009, 1700,  870,   0,  460,  310,   36,   0),
    (2010, 2063,  894,   0,  476,  531,  153,   0),
    (2011, 2342,  931,   0,  476,  643,  281,   0),
    (2012, 2494,  954,   0,  476,  709,  355,   0),
    (2013, 2489,  953,   0,  466,  701,  369,   0),
    (2014, 2613,  995,   0,  469,  738,  406,   0),
    (2015, 2827, 1081,   0,  471,  772,  492,   0),
    (2016, 3052, 1182,   0,  465,  862,  593,   0),
    (2017, 3485, 1396,   0,  457,  895,  737,   0),
    (2018, 3584, 1450,   0,  457,  924,  753,   0),
    (2019, 3777, 1572,  18,  457,  936,  772,   0),
    (2020, 4154, 1621, 179,  459,  960,  874,   0),
    (2021, 4697, 1798, 362, 1495,    0, 1000, 100),
    (2022, 5079, 1954, 499, 1495,    0, 1077, 162),
    (2023, 5346, 2066, 572, 1495,    0, 1183, 237),
    (2024, 5392, 2088, 589, 1495,    0, 1241, 261),
    (2025, 5477, 2110, 610, 1495,    0, 1340, 278),
    (2026, 5496, 2127, 628, 1495,    0, 1381, 298),
]

cols = ['year','total','sse_main','sse_star','szse_main','szse_sme','szse_gem','bse']
df_annual = pd.DataFrame(annual_records, columns=cols)
df_annual['sse_total']  = df_annual['sse_main']  + df_annual['sse_star']
df_annual['szse_total'] = df_annual['szse_main'] + df_annual['szse_sme'] + df_annual['szse_gem']
df_annual['yoy_growth'] = df_annual['total'].pct_change() * 100
df_annual['yoy_added']  = df_annual['total'].diff()

df_annual.to_csv('data/annual_totals.csv', index=False, encoding='utf-8-sig')
print(f'✅ 年度数据加载完成，共 {len(df_annual)} 年')
display(df_annual[['year','total','sse_main','sse_star','sse_total',
                    'szse_main','szse_sme','szse_gem','szse_total','bse']].tail(8))


### 2.2 各交易所最新概况（2026年3月）

In [ ]:
exchange_overview = pd.DataFrame([
    {'交易所': '上海证券交易所(上交所)', '合计': 2755, '主板': 2127, '科创板': 628,  '中小板': '—', '创业板': '—', '数据日期': '2026-03-12'},
    {'交易所': '深圳证券交易所(深交所)', '合计': 2876, '主板': 1495, '科创板': '—',  '中小板': '—', '创业板': 1381,'数据日期': '2026-03-06'},
    {'交易所': '北京证券交易所(北交所)', '合计': 298,  '主板': '—',  '科创板': '—',  '中小板': '—', '创业板': '—', '数据日期': '2026-03-13'},
])
exchange_overview.to_csv('data/exchange_overview.csv', index=False, encoding='utf-8-sig')
print('✅ 交易所概况数据加载完成')
display(exchange_overview)


### 2.3 行业分布数据（2026年1月，证监会行业分类）

In [ ]:
industry_data = {
    '上交所': [
        ('制造业',               1452),
        ('信息传输、软件和信息技术服务业', 145),
        ('电力、热力、燃气及水生产和供应业', 120),
        ('交通运输、仓储和邮政业',   110),
        ('金融业',                82),
        ('批发和零售业',           85),
        ('采矿业',                78),
        ('建筑业',                54),
        ('房地产业',              68),
        ('水利、环境和公共设施管理业', 42),
        ('科学研究和技术服务业',    38),
        ('租赁和商务服务业',        28),
        ('农、林、牧、渔业',        24),
        ('卫生和社会工作',          16),
        ('文化、体育和娱乐业',      14),
        ('其他',                  49),
    ],
    '深交所': [
        ('制造业',               1852),
        ('信息传输、软件和信息技术服务业', 285),
        ('批发和零售业',          165),
        ('金融业',                35),
        ('房地产业',              78),
        ('建筑业',                62),
        ('电力、热力、燃气及水生产和供应业', 42),
        ('交通运输、仓储和邮政业',   55),
        ('采矿业',                28),
        ('科学研究和技术服务业',    68),
        ('农、林、牧、渔业',        32),
        ('租赁和商务服务业',        45),
        ('水利、环境和公共设施管理业', 38),
        ('卫生和社会工作',          21),
        ('文化、体育和娱乐业',      30),
        ('其他',                  44),
    ],
    '北交所': [
        ('机械设备',              62),
        ('基础化工',              30),
        ('电力设备',              33),
        ('医药生物',              27),
        ('汽车',                  25),
        ('计算机',                24),
        ('信息技术服务',          28),
        ('电子',                  19),
        ('新材料',                22),
        ('农林牧渔',              14),
        ('食品饮料',              10),
        ('轻工制造',               9),
        ('其他',                  15),
    ],
}

dfs = []
for exch, rows in industry_data.items():
    tmp = pd.DataFrame(rows, columns=['行业', '数量'])
    tmp['交易所'] = exch
    tmp['占比(%)'] = (tmp['数量'] / tmp['数量'].sum() * 100).round(2)
    dfs.append(tmp)
df_industry = pd.concat(dfs, ignore_index=True)
df_industry.to_csv('data/industry_distribution.csv', index=False, encoding='utf-8-sig')
print(f'✅ 行业分布数据加载完成，共 {len(df_industry)} 条记录')


## 3. 统计分析

### 3.1 关键指标汇总

In [ ]:
latest = df_annual[df_annual['year'] == 2026].iloc[0]
base   = df_annual[df_annual['year'] == 1990].iloc[0]
decade = df_annual[df_annual['year'] == 2016].iloc[0]

growth_30yr  = (latest['total'] / base['total'] - 1) * 100
growth_10yr  = (latest['total'] / decade['total'] - 1) * 100
avg_new      = df_annual['yoy_added'].mean()
peak_year    = df_annual.loc[df_annual['yoy_added'].idxmax(), 'year']
peak_added   = df_annual['yoy_added'].max()
avg_rate     = df_annual['yoy_growth'].dropna().mean()

print('=' * 62)
print('  📊  中国股票市场核心统计指标（截至2026年3月）')
print('=' * 62)
print(f'  上市公司总数          : {int(latest["total"]):,} 家')
print(f'    上交所（主板+科创板）: {int(latest["sse_total"]):,} 家')
print(f'      其中 主板          : {int(latest["sse_main"]):,} 家')
print(f'      其中 科创板        : {int(latest["sse_star"]):,} 家')
print(f'    深交所（主板+创业板）: {int(latest["szse_total"]):,} 家')
print(f'      其中 主板          : {int(latest["szse_main"]):,} 家')
print(f'      其中 创业板        : {int(latest["szse_gem"]):,} 家')
print(f'    北交所               : {int(latest["bse"]):,} 家')
print(f'  30年累计增长           : {growth_30yr:.0f}%（{int(base["total"])} → {int(latest["total"])}）')
print(f'  近10年累计增长         : {growth_10yr:.0f}%')
print(f'  年均新增上市公司       : {avg_new:.0f} 家/年')
print(f'  单年新增最多           : {int(peak_year)} 年（新增 {int(peak_added):,} 家）')
print(f'  年均增长率（1991-2026）: {avg_rate:.2f}%')
print('=' * 62)


### 3.2 阶段性分析

In [ ]:
stages = [
    ('起步期',       1990, 1999),
    ('调整整合期',   2000, 2008),
    ('快速扩容期',   2009, 2019),
    ('多元发展期',   2020, 2026),
]

print(f'  {"阶段":<10} {"起止年份":<12} {"期初":>6} {"期末":>6} {"净增":>6} {"增幅":>8} {"CAGR":>8}')
print('  ' + '-' * 62)
for name, y0, y1 in stages:
    r0 = df_annual[df_annual['year'] == y0].iloc[0]
    r1 = df_annual[df_annual['year'] == y1].iloc[0]
    net  = int(r1['total'] - r0['total'])
    gain = (r1['total'] / r0['total'] - 1) * 100
    cagr = ((r1['total'] / r0['total']) ** (1/(y1-y0)) - 1) * 100
    print(f'  {name:<10} {y0}–{y1:<7} {int(r0["total"]):>6} {int(r1["total"]):>6} {net:>6} {gain:>7.1f}% {cagr:>7.2f}%')


### 3.3 行业集中度分析（HHI指数）

In [ ]:
print('各交易所行业集中度（HHI 指数，满分10000，越高表示越集中）\n')
for exch in ['上交所', '深交所', '北交所']:
    tmp = df_industry[df_industry['交易所'] == exch].copy()
    shares = tmp['数量'] / tmp['数量'].sum()
    hhi = (shares ** 2).sum() * 10000
    top3 = tmp.nlargest(3, '数量')[['行业','数量','占比(%)']].reset_index(drop=True)
    mfg  = tmp[tmp['行业'].str.contains('制造|机械|化工|电力设备')]['占比(%)'].sum()
    print(f'  【{exch}】  HHI = {hhi:.0f}  |  制造类合计占比 ≈ {mfg:.1f}%')
    for i, r in top3.iterrows():
        bar = '█' * int(r['占比(%)'] / 2)
        print(f'    {i+1}. {r["行业"]:<24} {int(r["数量"]):>5}家  {r["占比(%)"]:.1f}%  {bar}')
    print()


## 4. 数据可视化

### 4.1 年度上市公司总数趋势（1990–2026）

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

palette = {'sse_total': '#E74C3C', 'szse_total': '#3498DB', 'bse': '#27AE60'}
labels  = {'sse_total': '上交所', 'szse_total': '深交所', 'bse': '北交所'}

for col, color in palette.items():
    ax.plot(df_annual['year'].to_numpy(), df_annual[col].to_numpy(), marker='o', markersize=3,
            linewidth=2, color=color, label=labels[col])

ax.fill_between(df_annual['year'].to_numpy(), df_annual['total'].to_numpy(), alpha=0.05, color='#2C3E50')
ax.plot(df_annual['year'].to_numpy(), df_annual['total'].to_numpy(), linewidth=2.5, linestyle='--',
        color='#2C3E50', label='三所合计', zorder=5)

keypoints = [(1990, '10'), (2000, '1,088'), (2010, '2,063'), (2020, '4,154'), (2026, '5,496')]
for yr, lbl in keypoints:
    val = df_annual[df_annual['year'] == yr]['total'].values[0]
    ax.annotate(lbl, (yr, val), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=8.5, color='#2C3E50',
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=0.8))

ax.axvline(2021.9, color='#27AE60', linewidth=1, linestyle=':', alpha=0.7)
ax.text(2022.0, 300, '北交所\n开市', fontsize=8, color='#27AE60')
ax.axvline(2019.6, color='#E74C3C', linewidth=1, linestyle=':', alpha=0.5)
ax.text(2019.65, 300, '科创板\n开市', fontsize=8, color='#E74C3C')

ax.set_title('中国上市公司数量年度变化趋势（1990–2026）', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('年份', fontsize=11)
ax.set_ylabel('上市公司数量（家）', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.25, linestyle='--')
ax.set_xlim(1989, 2027)
plt.tight_layout()
plt.savefig('output/figures/01_annual_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 图1 已保存')


### 4.2 各交易所分板块堆叠柱状图

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

stack_cols   = ['sse_main', 'sse_star', 'szse_main', 'szse_sme', 'szse_gem', 'bse']
stack_colors = ['#C0392B', '#E74C3C', '#1A5276', '#2E86C1', '#85C1E9', '#27AE60']
stack_labels = ['上交所·主板', '上交所·科创板', '深交所·主板', '深交所·中小板(2004-2020)', '深交所·创业板', '北交所']

# 全部转为 numpy array，避免 pandas Series 引发多维索引错误
years  = df_annual['year'].to_numpy(dtype=float)
bottom = np.zeros(len(df_annual), dtype=float)
for col, color, label in zip(stack_cols, stack_colors, stack_labels):
    vals = df_annual[col].to_numpy(dtype=float)
    ax.bar(years, vals, bottom=bottom, color=color, label=label, width=0.75, alpha=0.88)
    bottom += vals

ax.annotate('2021年中小板\n并入主板', xy=(2021, 2200), xytext=(2015, 2800),
            arrowprops=dict(arrowstyle='->', color='#555'), fontsize=8, color='#555')
ax.set_title('三大交易所各板块上市公司数量（堆叠图，1990–2026）', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('年份', fontsize=11)
ax.set_ylabel('上市公司数量（家）', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.grid(True, alpha=0.2, axis='y', linestyle='--')
plt.tight_layout()
plt.savefig('output/figures/02_exchange_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 图2 已保存')


### 4.3 最新各交易所及板块占比饼图（2026年3月）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
latest_row = df_annual[df_annual['year'] == 2026].iloc[0]

# 三所占比
ev = [latest_row['sse_total'], latest_row['szse_total'], latest_row['bse']]
el = [f'上交所\n{int(ev[0]):,}家', f'深交所\n{int(ev[1]):,}家', f'北交所\n{int(ev[2]):,}家']
axes[0].pie(ev, labels=el, autopct='%1.1f%%', colors=['#E74C3C','#3498DB','#27AE60'],
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('三大交易所占比（2026年3月）', fontsize=12, fontweight='bold')

# 上交所板块
sv = [latest_row['sse_main'], latest_row['sse_star']]
sl = [f'主板\n{int(sv[0]):,}家', f'科创板\n{int(sv[1]):,}家']
axes[1].pie(sv, labels=sl, autopct='%1.1f%%', colors=['#C0392B','#F1948A'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('上交所板块构成', fontsize=12, fontweight='bold')

# 深交所板块
dv = [latest_row['szse_main'], latest_row['szse_gem']]
dl = [f'主板\n{int(dv[0]):,}家', f'创业板\n{int(dv[1]):,}家']
axes[2].pie(dv, labels=dl, autopct='%1.1f%%', colors=['#1A5276','#85C1E9'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[2].set_title('深交所板块构成', fontsize=12, fontweight='bold')

plt.suptitle('2026年3月各交易所及板块占比', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/figures/03_exchange_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 图3 已保存')


### 4.4–4.6 各交易所行业分布

In [ ]:
def plot_industry(exch_name, color_base, filename, fig_title):
    tmp = (df_industry[df_industry['交易所'] == exch_name]
           .sort_values('数量', ascending=True)
           .copy()
           .reset_index(drop=True))
    n = len(tmp)
    cmap   = plt.cm.get_cmap('RdYlGn')
    colors = [cmap(float(x)) for x in np.linspace(0.2, 0.85, n)]

    # 全部转为 Python list，彻底规避 Series → matplotlib 兼容问题
    industries = tmp['行业'].tolist()
    counts     = tmp['数量'].tolist()
    pcts       = tmp['占比(%)'].tolist()
    max_count  = float(max(counts))

    fig, ax = plt.subplots(figsize=(11, max(6, n * 0.52)))
    bars = ax.barh(industries, counts, color=colors, edgecolor='white', linewidth=0.5)

    for bar, val, pct in zip(bars, counts, pcts):
        ax.text(bar.get_width() + max_count * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{int(val):,}家 ({pct:.1f}%)', va='center', fontsize=9)

    ax.set_title(fig_title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('上市公司数量（家）', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.set_xlim(0, max_count * 1.22)
    ax.grid(True, alpha=0.2, axis='x', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'output/figures/{filename}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ {fig_title} 已保存')

plot_industry('上交所', '#E74C3C', '04_industry_sse.png',  '上交所行业分布（2026年1月）')
plot_industry('深交所', '#3498DB', '05_industry_szse.png', '深交所行业分布（2026年1月）')
plot_industry('北交所', '#27AE60', '06_industry_bse.png',  '北交所行业分布（2026年1月）')


### 4.7 年度增长分析

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [1, 1.2]})

df_plot  = df_annual.dropna(subset=['yoy_added']).copy()
years_p  = df_plot['year'].to_numpy(dtype=float)
added_p  = df_plot['yoy_added'].to_numpy(dtype=float)
bar_colors = ['#E74C3C' if v < 0 else '#27AE60' for v in added_p]

ax1.bar(years_p, added_p, color=bar_colors, width=0.7, alpha=0.85)
ax1.axhline(0, color='black', linewidth=0.8)

peak_idx = int(np.argmax(added_p))
ax1.annotate(
    f'峰值：{int(years_p[peak_idx])}年\n+{int(added_p[peak_idx])}家',
    xy=(years_p[peak_idx], added_p[peak_idx]),
    xytext=(years_p[peak_idx] - 5, added_p[peak_idx] - 100),
    arrowprops=dict(arrowstyle='->', color='#E74C3C'), fontsize=9, color='#E74C3C'
)
ax1.set_title('年度新增上市公司数量（1991–2026）', fontsize=12, fontweight='bold')
ax1.set_ylabel('当年新增家数', fontsize=10)
ax1.grid(True, alpha=0.2, axis='y', linestyle='--')
ax1.set_xlim(1989.5, 2026.5)

years_a = df_annual['year'].to_numpy(dtype=float)
for col, color, label in [('sse_total','#E74C3C','上交所'), ('szse_total','#3498DB','深交所'), ('bse','#27AE60','北交所')]:
    ax2.plot(years_a, df_annual[col].to_numpy(dtype=float),
             marker='o', markersize=3, linewidth=1.8, color=color, label=label)

ax2.set_title('三大交易所上市公司数量历史走势', fontsize=12, fontweight='bold')
ax2.set_xlabel('年份', fontsize=10)
ax2.set_ylabel('上市公司数量（家）', fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.2, linestyle='--')
ax2.set_xlim(1989.5, 2026.5)

plt.tight_layout()
plt.savefig('output/figures/07_growth_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 图7 已保存')


## 5. 汇总表格展示

In [ ]:
print('【表1】三大交易所最新概况（2026年3月）')
display(exchange_overview)

print('\n【表2】近六年上市公司数量变化')
recent = df_annual[df_annual['year'] >= 2021][[
    'year','total','sse_main','sse_star','sse_total','szse_main','szse_gem','szse_total','bse','yoy_added'
]].copy()
recent.columns = ['年份','合计','上交-主板','上交-科创','上交合计','深交-主板','深交-创业','深交合计','北交所','当年新增']
display(recent.reset_index(drop=True))

print('\n【表3】各交易所前5大行业（2026年1月）')
top5_all = []
for exch in ['上交所','深交所','北交所']:
    top5 = df_industry[df_industry['交易所']==exch].nlargest(5,'数量')[['交易所','行业','数量','占比(%)']]
    top5_all.append(top5)
display(pd.concat(top5_all).reset_index(drop=True))


## 6. 导出 Excel 综合报告

In [ ]:
with pd.ExcelWriter('output/reports/stock_market_report.xlsx', engine='openpyxl') as writer:

    # 年度历史数据
    out_annual = df_annual.rename(columns={
        'year':'年份','total':'合计','sse_main':'上交-主板','sse_star':'上交-科创板',
        'sse_total':'上交所合计','szse_main':'深交-主板','szse_sme':'深交-中小板',
        'szse_gem':'深交-创业板','szse_total':'深交所合计','bse':'北交所',
        'yoy_growth':'同比增长率(%)','yoy_added':'当年新增'
    })
    out_annual.to_excel(writer, sheet_name='年度历史数据', index=False)

    # 交易所概况
    exchange_overview.to_excel(writer, sheet_name='交易所概况', index=False)

    # 各交所行业分布
    for exch in ['上交所','深交所','北交所']:
        tmp = df_industry[df_industry['交易所'] == exch].drop(columns='交易所')
        tmp.to_excel(writer, sheet_name=f'{exch}行业分布', index=False)

    # 统计摘要
    latest_r = df_annual[df_annual['year'] == 2026].iloc[0]
    summary = pd.DataFrame([
        {'指标': '上市公司总数（2026年3月）',    '数值': f"{int(latest_r['total']):,} 家"},
        {'指标': '上交所（主板+科创板）',         '数值': f"{int(latest_r['sse_total']):,} 家"},
        {'指标': '  其中 主板',                  '数值': f"{int(latest_r['sse_main']):,} 家"},
        {'指标': '  其中 科创板',                '数值': f"{int(latest_r['sse_star']):,} 家"},
        {'指标': '深交所（主板+创业板）',         '数值': f"{int(latest_r['szse_total']):,} 家"},
        {'指标': '  其中 主板',                  '数值': f"{int(latest_r['szse_main']):,} 家"},
        {'指标': '  其中 创业板',                '数值': f"{int(latest_r['szse_gem']):,} 家"},
        {'指标': '北交所',                       '数值': f"{int(latest_r['bse']):,} 家"},
        {'指标': '上交所占比',                   '数值': f"{latest_r['sse_total']/latest_r['total']*100:.1f}%"},
        {'指标': '深交所占比',                   '数值': f"{latest_r['szse_total']/latest_r['total']*100:.1f}%"},
        {'指标': '北交所占比',                   '数值': f"{latest_r['bse']/latest_r['total']*100:.1f}%"},
        {'指标': '年均增长率（1991–2026）',      '数值': f"{df_annual['yoy_growth'].dropna().mean():.2f}%"},
        {'指标': '数据截止日期',                 '数值': '2026年3月'},
    ])
    summary.to_excel(writer, sheet_name='统计摘要', index=False)

print('✅ Excel报告已生成：output/reports/stock_market_report.xlsx')
print('   工作表：年度历史数据 / 交易所概况 / 上交所行业分布 / 深交所行业分布 / 北交所行业分布 / 统计摘要')


## 7. 分析结论

### 📈 宏观趋势
- **35年增长超548倍**：从1990年10家增长至2026年约5,496家，中国已成为全球第二大股票市场。
- **增长节奏分化**：2009–2019年是扩容最快十年，年均新增约200家；注册制全面推行后，IPO节奏趋于有序稳定。
- **历史峰值**：2017年单年新增最多（约343家），受益于供给侧改革与资本市场改革红利。

### 🏛️ 交易所格局
- **深交所**规模最大，占全市场约52%，是创业型与中小型企业核心聚集地；创业板是近年扩容主力。
- **上交所**主要承载大型蓝筹及科技创新龙头，科创板自2019年开市后累计汇聚628家硬科技企业。
- **北交所**自2021年11月开市以来，仅4年已汇聚298家专精特新企业，是三板市场转型升级的重要成果。

### 🏭 行业分布
- **制造业高度主导**：三所制造业均排名第一，深交所占比超64%，上交所约53%，体现中国制造业在资本市场的核心地位。
- **信息技术快速崛起**：深交所信息技术类占比约10%，是制造业外第二大行业，反映数字经济转型趋势。
- **北交所结构多元**：机械、化工、电力设备、医药等传统与新兴赛道并存，专精特新特色鲜明，集中度相对较低（HHI最低）。

### ⚠️ 数据说明
| 项目 | 说明 |
|---|---|
| 中小板 | 2004年设立，2021年6月并入深交所主板，不再单独统计 |
| 科创板 | 2019年7月开市，主要面向科技创新型企业 |
| 北交所 | 2021年11月正式开市，依托新三板精选层 |
| 2026年数据 | 截至2026年3月，全年数据以官方年报为准 |
